# SHAP Exercise:
#### **How Do Different SHAP Methods Explain the Same Image?**

SHAP has different **explainers** that use different strategies to compute the Shapley values. In this exercise, you'll apply different explainers to the same image and compare their explanations.

## Setup and Imports
Start by setting up a new virtual environment with the packages specified in requirements_shap.txt.
I tested the scripts with python 3.11 but newer versions probably also work.

Then download the *poodle.png* and *imagenet_classes.txt* from Moodle into a folder called Data. Since we will reuse this folder, you can create it one level above the folder for this SHAP exercise.

Then:

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
import requests
from skimage.segmentation import slic
import shap
import cv2

device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
# Load pre-trained VGG16
model = models.vgg16(weights="IMAGENET1K_V1")
model.eval()
model = model.to(device)

# Load ImageNet class names
url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
response = requests.get(url)
class_names = [line.strip() for line in response.text.split('\n')]
print(f"Model loaded. {len(class_names)} classes.")

In [ ]:
# Path to images
data_path = Path("../Data")
img_name = "poodle.png"

# Load image
img_path = data_path / img_name
if img_path.exists():
    original_img = Image.open(img_path).convert('RGB')
else:
    print("Image not found, creating placeholder")
    original_img = Image.new('RGB', (224, 224), color='gray')

original_img = original_img.resize((224, 224))
img_array = np.array(original_img)

plt.imshow(original_img)
plt.title(f"Test Image: {img_name}")
plt.axis('off')
plt.show()

In [ ]:
# Preprocessing for Imagenet like data
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# Get model prediction
img_tensor = preprocess(original_img).unsqueeze(0).to(device)
with torch.no_grad():
    output = model(img_tensor)
    probs = torch.nn.functional.softmax(output[0], dim=0)
    top5 = torch.argsort(probs, descending=True)[:5]

predicted_class = top5[0].item()
print(f"\nPredicted: {class_names[predicted_class]} ({probs[predicted_class]:.3f})")
print("\nTop 5 predictions:")
for i, idx in enumerate(top5):
    print(f"  {i+1}. {class_names[idx]}: {probs[idx]:.3f}")

## Prediction Function
The SHAP library implements different variations of SHAP in functions called *explainers*.
All SHAP explainers need a function that takes images and returns predictions.

In [ ]:
def model_predict(images):
    """
    Predict class probabilities for a batch of images.
    
    Args:
        images: numpy array of shape (n_samples, H, W, 3) with values in [0, 255]
    
    Returns:
        numpy array of shape (n_samples, n_classes) with probabilities
    """
    # TODO: Implement this function
    # YOUR CODE HERE
    
    return probs.cpu().numpy()

# Test the function
test_input = np.array([img_array])
test_output = model_predict(test_input)
print(f"Test output shape: {test_output.shape}")
print(f"Probability for {class_names[predicted_class]}: {test_output[0, predicted_class]:.3f}")

# KernelSHAP
We will start with the basic KernelSHAP explainer which we saw in the lecture.

**Reminder how it works**: Kernel SHAP approximates Shapley values by sampling coalitions of features (here, superpixels) and weighting them with a Shapley kernel; it is model‑agnostic but relatively slow.

## Masking

The SHAP library needs a masking function to represent "feature absent" perturbation. 
For our image data and KernelSHAP, we will use a superpixel segmentation approach similar to LIME, as discussed in the lecture. 
To this end, we first segement the image using the SLIC algorithm. If you want, you can play around with the hyperparameters of SLIC.

In [ ]:
# Superpixels
n_segments = 50 # Number of superpixels
segments = slic(img_array, n_segments=n_segments, compactness=30, sigma=3, start_label=0) # This returns a 2D integer array assigning a superpixel id to each pixel.


In addition, SHAP requires a background data for perturbation. In our case this background data simply defines the "color" that is used to perturb the superpixels.
Let's start with a simple perturbation using only black color:

In [ ]:
background = np.zeros((1, 224, 224, 3), dtype=np.uint8)

Now, we can create custom masking functions to perturb the image and get the models prediction on this perturbed sample. 

In [ ]:
def mask_image_kernel(masks, segmentation, image, background):
    '''
    A utility masking function for Kernel SHAP that converts binary coalition masks
    over superpixels into perturbed images.

    :param masks: A 2D binary array of shape (n_masks, n_segments) where the entries of each mask
        indicate which superpixels are kept for the masked sample.
    :param segmentation: A 2D integer array assigning each pixel to a superpixel id.
        Superpixel ids are expected to match the column indices of `masks`.
    :param image: The original input image of shape (H, W, 3).
    :param background: The baseline image used to fill masked-out superpixels.
        Must have the same shape as `image`.
    :return: A batch of perturbed images of shape (n_masks, H, W, 3), where each
        image corresponds to one coalition mask.
    '''
    # TODO YOUR CODE HERE

def f_kernel(masks):
    '''
    A utility prediction function that uses binary masks to predict different coalitions.
    :param masks: A binary mask [0,1,0,...] that represents which superpixels are present.
    :return: The classification for the perturbed image.
    '''
    # TODO YOUR CODE HERE

## KernelExplainer

Now that we have the necessary auxiliary functions, let's create our SHAP explanation.
Use the *KernelExplainer* class and its *shap_values* method to explain our poodle image.
Hint: The KernelExplainer expects the background argument in binary coalition mask form.
- See SHAP docs: [shap.KernelExplainer](https://shap.readthedocs.io/en/latest/generated/shap.KernelExplainer.html)


In [ ]:
# TODO: Create and run KernelExplainer
# YOUR CODE HERE

# TODO: Extract shap values for predicted class
# YOUR CODE HERE

Now, use the shap.image_plot method to visualize your result. 
Hint: you might need another helper function.
- See SHAP docs: [shap.image_plot]( https://shap.readthedocs.io/en/latest/generated/shap.plots.image.html)

In [ ]:
# Visualize KernelExplainer SHAP values
# TODO YOUR CODE HERE

## Different Background
Now lets try a random background for perturbation instead of black. 
Create a background image with random pixels (an image where each pixel has random values between 0 and 255) and use this for our custom masking function.
Then, see how this changes the explanation for your example image.

In [ ]:
# TODO YOUR CODE HERE 

# PartitionSHAP

Now, we will look at a different way to calculate SHAP values: the PartitionExplainer.

**How it works**: Uses a hierarchical feature partition tree and a masking function (for images, shap.maskers.Image) to efficiently approximate Shapley values, making it much faster than kernel-based methods for structured inputs.
See SHAP docs: 
- [shap.PartitionExplainer](https://shap.readthedocs.io/en/latest/generated/shap.PartitionExplainer.html)
- [shap.maskers.Image](https://shap.readthedocs.io/en/latest/generated/shap.maskers.Image.html)

## ParitionExplainer with Inpainting

In the first PartitionExplainer example, we will use inpainting to mask the images. Here the masking tries to reconstruct the selected image area from the pixels near the area boundary. This leads to more realistic perturbations than using a constant color.

In [ ]:
# Image masker
masker = shap.maskers.Image("inpaint_telea", img_array.shape)

# TODO: Create Partition explainer
# YOUR CODE HERE

# TODO: Compute SHAP values
# YOUR CODE HERE

In [ ]:
# TODO: Visualize PartitionExplainer SHAP values using shap.image_plot
# YOUR CODE HERE

## PartitionExplainer with Blur Masking

Now, let's try to use an image masker with a blur value (e.g. "blur(128,128)") so that masked regions are replaced by a blurred version of the image, representing “feature absent” via smoothing instead of inpainting or a fixed background.
- See SHAP docs: [shap.maskers.Image](https://shap.readthedocs.io/en/latest/generated/shap.maskers.Image.html)


In [ ]:
# TODO: Adjust the image masker from the previous example to use 128x128 blurring
# YOUR CODE HERE

# TODO: Create Partition explainer
# YOUR CODE HERE

# TODO: Compute SHAP values
# YOUR CODE HERE

In [ ]:
# TODO: Visualize PartitionExplainer SHAP values using shap.image_plot
# YOUR CODE HERE